# Otimização de Coluna Tubular à Flambagem

Tradução em Python dos scripts MATLAB da pasta `Ex2`
(`Ex2_flambagem_coluna.m`, `massa_coluna.m`, `tensao_maxima.m`).

Dimensionamento de uma coluna tubular de parede fina (raio $R=x_1$, espessura $t=x_2$,
comprimento $L$), minimizando sua massa sujeita à condição de não-flambagem sob uma carga
de compressão axial $P$ (carga crítica de Euler para um tubo de parede fina,
$I\approx\pi R^3 t$):
$$\min_{R,t}\; \rho L\,(2\pi R t) \quad \text{s.a.} \quad P \le P_{cr}=\dfrac{\pi^3 E R^3 t}{4L^2},\;\; x_{min}\le x \le x_{max}$$

In [1]:
import numpy as np
from scipy.optimize import minimize

## Função objetivo e restrição (`massa_coluna.m`, `tensao_maxima.m`)

In [2]:
rho = 2700       # kg/m^3
comprimento = 1  # m
P = 1e12         # N (carga de compressão)
E = 70e9         # Pa (módulo de elasticidade)

xmin = [0.02, 0.005]
xmax = [0.5, 0.05]


def massa_coluna(x):
    R, t = x
    return rho * comprimento * 2 * np.pi * R * t


def tensao_maxima(x):
    """g(x) >= 0 <=> P <= Pcr (carga crítica de flambagem de Euler)."""
    R, t = x
    Pcr = (np.pi ** 3 * E * R ** 3 * t) / (4 * comprimento ** 2)
    return np.array([Pcr - P])

## Otimização (`Ex2_flambagem_coluna.m`)

Chute inicial $x_0=[0.10,\,0.01]$ m. **Atenção**: com $P=10^{12}$ N — uma carga
extraordinariamente alta —, mesmo a maior coluna permitida pelos limites
($R=0.5$ m, $t=0.05$ m) não atinge a carga crítica necessária; o problema, como
especificado, é **infactível** dentro dos limites de projeto. O otimizador converge para o
limite superior de tamanho sem satisfazer plenamente a restrição — reproduzindo fielmente
o comportamento que o `fmincon` do script original também apresentaria (não-convergência /
aviso de restrição violada).

In [3]:
x0 = [0.10, 0.01]

res = minimize(
    massa_coluna, x0, method='SLSQP',
    bounds=list(zip(xmin, xmax)),
    constraints={'type': 'ineq', 'fun': tensao_maxima},
    options={'maxiter': 300, 'ftol': 1e-12},
)

R_opt, t_opt = res.x
Pcr_opt = (np.pi ** 3 * E * R_opt ** 3 * t_opt) / (4 * comprimento ** 2)

print('Convergiu:', res.success, '-', res.message)
print('x = [R, t] =', res.x)
print('massa =', res.fun)
print('Pcr =', Pcr_opt, ' (necessário P =', P, ')')

Convergiu: False - Positive directional derivative for linesearch
x = [R, t] = [0.11754004 0.005     ]
massa = 9.970099028350575
Pcr = 4405702.590245289  (necessário P = 1000000000000.0 )
